In [1]:
from debugbox import DOSBoxDebugger
from pathlib import Path
import time
from collections import Counter
import subprocess

In [4]:
appname='/opt/homebrew/bin/dosbox-x'

def activate(app):
    subprocess.run(["osascript", "-e", f'tell application "System Events" to set frontmost of process "{app}" to true'])
def send_key_code(code, app=appname, modifier=None):
    subprocess.run(["osascript", "-e", f'tell application "{appname}" to activate'])
    mod = f" using {{{modifier}}}" if modifier else ""
    script = f'''
      tell application "System Events"
      tell process "{app}"
          key code {code}{mod}
      end tell
      end tell'''
    subprocess.run(["osascript", "-e", script])

In [11]:
import time

import re

ANSI_RE = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
EV_RE = re.compile(r"LOG: EV of '([^']*)' is:\s*\n(?:LOG: )?([0-9a-fA-F]+(?:\s+[0-9a-fA-F]+)*)")

def get_registers(reg_names=("BX", "CS", "IP")):
    """
    Sends an EV command for the requested registers and reads the
    result back from the same logfile DOS MCBS writes to. Returns
    {reg_name: int_value}.
    """
    global last_pos
    new_text = dbg.ev(" ".join(reg_names))
    cleaned = ANSI_RE.sub('', new_text)
    m = EV_RE.search(cleaned)
    if not m:
        print('missing')
        return None
    names = m.group(1).split()
    values = [int(v, 16) for v in m.group(2).split()]
    ev_data = dict(zip(names, values))
    return ev_data


def run_until_break(reg_names=("BX", "CS", "IP"), max_wait=5.0, poll_interval=0.03):
    dbg.run()
    start = time.time()
    while time.time() - start < max_wait:
        regs = get_registers(reg_names)
        if regs is not None:
            return regs
        time.sleep(poll_interval)
    return None  # timed out -- worth logging/counting these separately

In [207]:
dbg = DOSBoxDebugger(
    conf_path="/Users/rmaxwell/Downloads/f19_2.conf",
    dump_path="/Users/rmaxwell/Documents/personal/MEMDUMP.BIN",
)
dbg.start()

AUTOTYPE ENTER
LOG: Early LOG Init complete
LOG: CONFIG: Loaded config file: /Users/rmaxwell/Downloads/f19_2.conf
LOG: DOSBox-X's working directory: /Users/rmaxwell/Documents/personal
LOG: Logging init: beginning logging proper. This is the end of the early init logging
LOG: Logging: opened logfile 'mcbs_log.txt' successfully. All further logging will go to this file.
LOG: DOSBox-X version 2026.08.02 Commit e522642 (macOS ARM SDL2 64-bit)
LOG: macOS EXE path: /opt/homebrew/bin/dosbox-x
LOG: macOS Resource path: 
LOG: macOS: NSTouchBar support detected in system
LOG: Prevent capture: 0
LOG: SDL: version 2.32.70, Video cocoa, Audio coreaudio
LOG: Host keyboard layout is now us (US English)
LOG: Mapper keyboard layout is now us (US English)
LOG: SDL2 reports desktop display mode 1512 x 982
LOG: The default output for the video system: opengl
LOG: OpenGL Version : 2.1 Metal - 90.5
LOG: Configured windowposition: -
LOG: SDL: Current window pixel format: SDL_PIXELFORMAT_ARGB8888
LOG: Releasi

In [145]:
dbg.send("BP 1CE8:00000E80")
# dbg.send("LOGC 100000")
dbg.run()
for i in range(10):
    send_key_code(10, app=appname)
    print('sent')


B_
P_
 _
1_C_E_8_:_0_0_0_0_0_E_8_0_LOG: DEBUG: Set breakpoint at 1CE8:0E80

_                
R_
U_
N_
(Running)                                                                      

0814:00000100 BC2607              mov  sp,0726                                  0814:00000103 BB7300              mov  bx,0073                                  000001030000072619000000 sent
sent
sent
sent
sent
sent
sent
sent
sent
sent
LOG: pixratio 1.200, dw true, dh false
LOG: Aspect ratio: 640 x 480  xToY=1.333 yToX=0.750
CA DB E8 1C.29 4F E8 1C)O00 47 E8 1CG
..

1CE8:00000E80 B83200              mov  ax,0032                                  1CE8:00000E83 F76EF2              imul word [bp-0E]           ss:[DA78]=0000    1CE8E86 8BF0si,ax
1CE8E88 037604add  si,[bp+04]ss:[DA8A]=571A
1CE8E8B 837C2E00cmp  word [si+2E],0000ds:[00C4]=006B
1CE8E8F 741Dje   00000EAE ($+1d)(no jmp)
I-> _                                                                          A096  DS=2CE8882CE8571A7D34  CS=1CE800000E80  C1 Z0 

In [204]:
#dbg.send('BPDEL *')
#dbg.send("BP 1CE8:00003D7F") -- once mission is loaded?
dbg.run()




7mN_
(Running)                                                                      

F000:0000E995 FA                  cli                                           F000:0000E996 B020                mov  al,20                                    0000E99620
06666667
LOG: DEBUG: Memory breakpoint : 0040:002A - 00 -> 0D


F000:0000E995 FA                  cli                                           F000:0000E996 B020                mov  al,20                                    
I-> _                                                                          CCB80000E9950 S1 O0 A1 P073525067278430518.306000000


In [162]:
def get_logcpu_file():
    p = Path("/Users/rmaxwell/Documents/personal/LOGCPU.TXT")
    text = p.read_text(errors="replace")
    return text

def get_pairs(text):
    pairs = [
        line.strip()
        for line in text.splitlines()
        if ":" in line
    ]
    counts = Counter(pairs)
    return counts

def get_transitions(text):
    pairs = [
        line.strip()
        for line in text.splitlines()
        if ":" in line
    ]
    transitions = Counter(
        zip(pairs[:-1], pairs[1:])
    )
    return transitions

#text = get_logcpu_file()
#pairs = get_pairs(text)
transitions = get_transitions(text)

In [1]:
#dbg.send("RUNWATCH")

In [ ]:
dbg.send("BP 1CE8:00000E80")

In [198]:
dbg.send("BP 1CE8:00000E80")
slots = [f"{off:04X}" for off in range(0x001E, 0x003E, 2)]  # 001E..003C, 16 word-slots

for off in slots:
    resp = dbg.send(f"BPM 0040:{off}")
    time.sleep(0.5)
dbg.run()


B_
P_
 _
1_C_E_8_:_0_0_0_0_0_E_8_0_LOG: DEBUG: Set breakpoint at 1CE8:0E80

_                
B_
P_
M_
 _0_0_4_0_:_0_0_1_E_LOG: DEBUG: Set memory breakpoint at 0040:001E

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_0_LOG: DEBUG: Set memory breakpoint at 0040:0020

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_2_LOG: DEBUG: Set memory breakpoint at 0040:0022

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_4_LOG: DEBUG: Set memory breakpoint at 0040:0024

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_6_LOG: DEBUG: Set memory breakpoint at 0040:0026

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_8_LOG: DEBUG: Set memory breakpoint at 0040:0028

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_A_LOG: DEBUG: Set memory breakpoint at 0040:002A

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_C_LOG: DEBUG: Set memory breakpoint at 0040:002C

_             
B_
P_
M_
 _0_0_4_0_:_0_0_2_E_LOG: DEBUG: Set memory breakpoint at 0040:002E

_             
B_
P_
M_
 _0_0_4_0_:_0_0_3_0_LOG: DEBUG: Set memory breakpoint at 0040:0030



In [195]:
dbg.send("EV CS DS ES SS IP SP")

 LOG: *** Debugger command not recognized37m
 0H





H_

F000:0000E995 FA                  cli                                           F000:0000E996 B020                mov  al,20                                    
(Running/watching)                                                             0000E9963
06666667
A5 FE 00 F0..

1CE8:00007650 8B0E7898            mov  cx,[9878]              ds:[9878]=0291    1CE8:00007654 BB7480              mov  bx,8074                                  1CE8:00007657 03D9add  bx,cx
1CE8:00007659 03D9add  bx,cx
1CE8:0000765B 03D9add  bx,cx
1CE8:0000765D 884702mov  [bx+02],alds:[0004]=0000   0000007400000FA081EB
00000002000001D8  CS=1CE87650  C0 Z0 S0 O0 A01
000000740000DA9C
000026570000DA94694750
95.000333333
99 B108    l,08                  9B 2ACB  sub  cl,bl  9D D3EAshr  dx,cl9F 3A1E7498cmp  bl,[9874]ds:[9874]=0B0BA3 7D14jge  000076B9 ($+14)(down)A5 268A05al,es:[di]es:[08D7]=277D   7208D799121
00FD2846259
846
44 8B1EBC98bx,[98BC]ds:[98BC]=000048 88877C98mov  [bx-6784],alds:[1A49]=00004C FF06BC98inc 

In [196]:
# enter from roster screen 0040:0026
# enter from mission screen 0040:0028 
# x from mission screen 0040:002A
# enter from mission screen 0040:002C
# x from briefing screen 0040:002E
# x from briefing screen 0040:0030
# x from briefing screen 0040:0032
# x from briefing screen 0040:0034
# x from briefing screen 0040:0036

dbg.run()

 U_
 LOG: *** Debugger command not recognized

 

In [178]:
#LOG: f000 d1f 888 d1f e995 1f78
#LOG: f000 d1f 888 d1f e995 1f78
dbg.ev('CS DS ES SS IP SP')


E_
V_
 _
C_S_ _D_S_ _E_S_ _S_S_ _I_P_ _S_P_LOG: EV of 'CS DS ES SS IP SP' is:
LOG: f000 d1f 888 d1f e995 1f78

_                    

"\x1b[10B\x1b[7B\n\x1b[75D\x1b\x1bE_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\n\x1b[74D\x1b\x1bV_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\n\x1b[73D\x1b\x1b _\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\n\x1b[72D\x1b\x1bC_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;9H\x1b\x1bS_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;10H\x1b\x1b _\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;11H\x1b\x1bD_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;12H\x1b\x1bS_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;13H\x1b\x1b _\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;14H\x1b\x1bE_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;15H\x1b\x1bS_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;16H\x1b\x1b _\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;17H\x1b\x1bS_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;18H\x1b\x1bS_\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7B\x1b[24;19H\x1b\x1b _\x1b[m\x1b[m\x1b\x1b\x1b[6;80H\x1b[10B\x1b[7

In [169]:
#dbg.send("BP 1CE8:00000E80")  -- down arrow press
#dbg.send("BP 1CE8:00001436")
#dbg.send("BP 1CE8:00001574")
dbg.send("BPM 0040:001C")

dbg.run()
send_key_code(45, app=appname)
time.sleep(1)
dbg.run()
send_key_code(18, app=appname)
time.sleep(1)


B_
P_
M_
 _0_0_4_0_:_0_0_1_C_LOG: DEBUG: Set memory breakpoint at 0040:001C

_             
R_
U_
N_
(Running)                                                                      


0814:00000103 BB7300              mov  bx,0073                                  0814:00000106 B44A                mov  ah,4A                                    000000730000010629333333 LOG: DEBUG: Memory breakpoint : 0040:001C - 00 -> 1E



0814:00000103 BB7300              mov  bx,0073                                  0814:00000106 B44A                mov  ah,4A                                    
I-> _                                                                          [210000007300000106666667 
R_
U_
N_
(Running)                                                                      



0814:00000106 B44A                mov  ah,4A                                    0814:00000108 CD21                int  21                                       00004A00000001083
30333333 LOG: DEBUG: Memory breakpoin

In [154]:
dbg.send("BP 1CE8:00000E80")

dbg.run()
send_key_code(45, app=appname)
time.sleep(1)
send_key_code(18, app=appname)
for i in range(5):
    send_key_code(36, app=appname)
    time.sleep(1)


B_
P_
 _
1_C_E_8_:_0_0_0_0_0_E_8_0_LOG: DEBUG: Set breakpoint at 1CE8:0E80

_                
R_
U_
N_
(Running)                                                                      

0814:00000100 BC2607              mov  sp,0726                                  0814:00000103 BB7300              mov  bx,0073                                  00000103000007261
666667 LOG: pixratio 1.200, dw true, dh false
LOG: Aspect ratio: 640 x 480  xToY=1.333 yToX=0.750


In [155]:
dbg.send("BPDEL *")
dbg.send("LOGC 1000000")
dbg.run()
send_key_code(36, app=appname)

CA DB E8 1C.29 4F E8 1C)O00 47 E8 1CG
..

1CE8:00000E80 B83200              mov  ax,0032                                  1CE8:00000E83 F76EF2              imul word [bp-0E]           ss:[DAAA]=0000    1CE8E86 8BF0si,ax
1CE8E88 037604add  si,[bp+04]ss:[DABC]=6E56
1CE8E8B 837C2E00cmp  word [si+2E],0000ds:[0092]=006B
1CE8E8F 741Dje   00000EAE ($+1d)(no jmp)
I-> _                                                                          4064  DS=2CE8882CE86E567D34  CS=1CE800000E80  C1 Z0 S1 O0 A1 P100DAB80000000DA8824103455885769.672000000


In [163]:
transitions

Counter({('1CE8:00009DB1', '1CE8:00009DAC'): 2890018,
         ('1CE8:00009DAC', '1CE8:00009DB1'): 2890015,
         ('F000:0000CFC0', 'F000:0000CFC1'): 12229,
         ('2CE8:00000CBC', '0A66:0000000A'): 12228,
         ('0A66:0000000A', '0A66:0000000C'): 12228,
         ('0A66:0000000C', 'F000:0000CFC0'): 12228,
         ('F000:0000CFC1', 'F000:0000CFC5'): 12228,
         ('1CE8:00001EB8', '1CE8:000061F1'): 12227,
         ('1CE8:000061F1', '1CE8:000061F6'): 12227,
         ('1CE8:00006207', '1CE8:00001EBB'): 12227,
         ('1CE8:000013DC', '2CE8:00000CBC'): 12227,
         ('F000:0000CFC5', '0A66:0000000E'): 12227,
         ('1CE8:000013E1', '1CE8:000013E3'): 12227,
         ('1CE8:00001598', '1CE8:00001EB8'): 12226,
         ('1CE8:00001EBB', '1CE8:000013DC'): 12226,
         ('0A66:00000013', '0A66:00000015'): 12226,
         ('0A66:00000017', '1CE8:000013E1'): 12226,
         ('1CE8:000013E3', '1CE8:000013E5'): 12226,
         ('1CE8:000013E5', '1CE8:000013E9'): 12226,
        

In [ ]:
def capture_logc_window(dbg, n_instructions=0x100000):
    dbg.send(f"LOGC {n_instructions:X}")
    dbg.send("BPDEL *")
    dbg.run()
    dbg.wait_for_log_complete()
    text = get_logcpu_file()
    return get_pairs(text), get_transitions(text)

screens = []
for label in ["roster", "mission_assignment",
              "briefing1", "targets", "briefing2", "ordnance", "menu"]:
    input(f"navigate to '{label}', then press Enter here to capture")
    pairs, transitions = capture_logc_window(dbg)
    screens.append((label, pairs, transitions))

target = "ordnance"
target_pairs, target_transitions = next((p, t) for l, p, t in screens if l == target)
other_addrs = set()
for label, pairs, _ in screens:
    if label != target:
        other_addrs |= set(pairs.keys())

unique_to_target = set(target_pairs.keys()) - other_addrs
print(f"{len(unique_to_target)} addresses seen ONLY on '{target}'")

entry_candidates = []
for edge, count in target_transitions.items():
    src, dst = edge.split(" -> ")
    if dst in unique_to_target and src in other_addrs:
        entry_candidates.append((edge, count))
entry_candidates.sort(key=lambda x: -x[1])
for edge, count in entry_candidates[:10]:
    print(edge, count)

navigate to 'roster', then press Enter here to capture 






LOG: DEBUG: Starting logm0_

g)                                                                      
I-> _                                                                          9000000

g)                                                                      
                                                                        
g)                                                                      
 _                                                                          
(Running)                                                                      
I-> _                                                                          3
4E36
33333333

(Running)                                                                      
I-> _                                                                          4
6D45
37666667

(Running)                                                                      
I-> _                                                                         

In [101]:
dbg.send("BPDEL *")
dbg.run()





LOG: DEBUG: Breakpoints deleted.

      


7mN_
(Running)                                                                      

1CE8:00006E54 D0E8                shr  al,1                                     1CE8:00006E56 D0E8                shr  al,1                                     00006E568
23666667


In [105]:
dbg.send("BPDEL *")
dbg.send("BPINT 21 4B")
dbg.run()





LOG: DEBUG: Breakpoints deleted.

      



LOG: DEBUG: Set interrupt breakpoint at INT 21 AH=4B

          


7mN_
(Running)                                                                      

1CE8:00006E54 D0E8                shr  al,1                                     1CE8:00006E56 D0E8                shr  al,1                                     00006E566
91333333
60 CA 00 F0`C0 D1 00 F0.
..

0814:00000259 CD21                int  21                                       0814:0000025B 8CC9                mov  cx,cs                                    0814:0000025D 8ED1mov  ss,cx
0814:0000025F 2E8B264705mov  sp,cs:[0547]cs:[0547]=0726
0814:00000264 8ED9mov  ds,cx
0814:00000266 8EC1mov  es,cx          
I-> _                                                                          4B100  DS=081408140814054F20814000002590814091C52507269789797338042.863000000


In [92]:
dbg.send("BP 2CE8:00000D11")
dbg.run()




7mN_
(Running)                                                                      

1CE8:00006E54 D0E8                shr  al,1                                     1CE8:00006E56 D0E8                shr  al,1                                     00006E566
13000000


1CE8:00006E54 D0E8                shr  al,1                                     1CE8:00006E56 D0E8                shr  al,1                                     
I-> _                                                                          3
400006E54E64
16666667

g)                                                                      
 _                                                                          
(Running)                                                                      
I-> _                                                                          4
6D73
21333333

(Running)                                                                      
I-> _                                                      

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)




(Running)                                                                      
I-> _                                                                          A
279
19666667

(Running)                                                                      
I-> _                                                                          B
46508
24000000

(Running)                                                                      
I-> _                                                                          C
6517
28333333

(Running)                                                                      
I-> _                                                                          D
8426
32666667

(Running)                                                                      
I-> _                                                                          E
A335
37000000

(Running)                                                                      
I-> _                                  

In [41]:
dbg.send("BPDEL *")
dbg.send("BP 1CE8:68A0")
logout = {}
target_hit_count = 4

for hit_num in range(target_hit_count):
    dbg.run()
    time.sleep(1)
    send_key_code(36, app=appname)  # Enter
    regs = get_registers(reg_names=('BX',))
    if regs:
        print(f"hit {hit_num}: BX={regs['BX']} ({regs['BX']*16} bytes)")
        logout[hit_num] = regs['BX']
    time.sleep(4)





LOG: DEBUG: Breakpoints deleted.

      



LOG: DEBUG: Set breakpoint at 1CE8:68A0

           


7mN_
(Running)                                                                      

1CE8:000068A0 CD21                int  21                                       F0000000D10000000DABA7
48333333


1CE8:000068A0 CD21                int  21                                       
I-> _                                                                          1
1011CE8000068A01FFFF
0000000DAC03939
742000000




LOG: EV of 'BX' is:X_
LOG: 101

hit 0: BX=257 (4112 bytes)



7mN_
(Running)                                                                      

1CE8:000068A0 CD21                int  21                                       F0000000D10000000DABA40
42666667
missing


KeyboardInterrupt: 

In [24]:
logout

{}